# 01 - Data Acquisition & Cleaning

**Project:** ARMA-GARCH Beta Risk Model  
**Author:** Amos Anderson  
**Purpose:** Download daily price data for all asset classes, verify data
integrity, compute log returns, and save processed data to `data/processed/`.

All reusable functions are implemented in `src/data_utils.py` and imported here.
The notebook documents decisions and shows diagnostic plots.

In [ ]:
import sys
from pathlib import Path

# Make src/ importable from the notebooks/ directory
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Plot style
plt.rcParams.update({
    "figure.dpi": 150,
    "figure.figsize": (12, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

print("Imports OK")

----

## Asset selection

Five asset classes are included to stress-test the model across different return distributions and volatility regimes:

| Ticker | Name | Asset class |
|--------|------|-------------|
| ^GSPC | S&P 500 | Equity index (required benchmark) |
| AAPL | Apple Inc. | Large-cap single stock |
| GLD | SPDR Gold ETF | Commodity proxy |
| TIP | iShares TIPS ETF | Inflation-protected fixed income |
| EURUSD=X | EUR/USD | Foreign exchange |

Date range: 2005-01-01 to 2025-12-31 (20 years, ~5,000 trading days).

Estimation window: 250 periods. Assessment window: 1,000 periods.

Minimum required: 1,250 periods. All assets exceed this comfortably.

In [ ]:
# Implement src/data_utils.py

from src.data_utils import (
    download_prices,
    check_corporate_actions,
    compute_log_returns,
    summary_statistics,
    save_processed,
    load_processed,
)
print("src.data_utils imported OK")

In [ ]:
# Download prices

TICKERS = ["^GSPC", "AAPL", "GLD", "TIP", "EURUSD=X"]
START   = "2005-01-01"
END     = "2025-12-31"

prices = download_prices(TICKERS, start=START, end=END)
prices.head()

In [ ]:
# Corporate action check

flags = check_corporate_actions(prices)

In [ ]:
# Missing data audit

missing = prices.isna().sum()
print("Missing values per ticker:")
print(missing)
print(f"\nTotal missing: {missing.sum()}")

In [ ]:
# Plot raw price series

fig, axes = plt.subplots(len(TICKERS), 1, figsize=(13, 14), sharex=True)

for ax, ticker in zip(axes, prices.columns):
    ax.plot(prices.index, prices[ticker], linewidth=0.8, color="#2c5f8a")
    ax.set_ylabel(ticker, fontsize=10)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

axes[-1].set_xlabel("Date")
fig.suptitle("Adjusted Close Prices — All Assets (2005–2025)", 
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../outputs/figures/01_raw_prices.png", bbox_inches="tight")
plt.show()
print("Figure saved to outputs/figures/01_raw_prices.png")

In [ ]:
# Compute log returns

returns = compute_log_returns(prices)
returns.head()

In [ ]:
# Plot return series

fig, axes = plt.subplots(len(TICKERS), 1, figsize=(13, 14), sharex=True)

for ax, ticker in zip(axes, returns.columns):
    ax.plot(returns.index, returns[ticker], linewidth=0.5, color="#2c5f8a", alpha=0.8)
    ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
    ax.set_ylabel(ticker, fontsize=10)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

axes[-1].set_xlabel("Date")
fig.suptitle("Daily Log Returns - All Assets (2005–2025)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../outputs/figures/01_log_returns.png", bbox_inches="tight")
plt.show()
print("Figure saved to outputs/figures/01_log_returns.png")

In [ ]:
# Summary Statistics

stats = summary_statistics(returns)
print("Summary Statistics\n")
print(stats.round(4).to_string())

---

## Interpreting the summary statistics

Key quantities to note before proceeding:

- **Excess kurtosis** -- all assets should show values well above 0 (the Gaussian
  baseline). Values of 3-15 are typical for daily equity returns. This is the
  primary empirical justification for using NIG over a Gaussian model.
- **Skewness** -- negative skewness (left tail heavier) is typical for equities,
  consistent with crash risk being larger than equivalent upside moves.
- **Volatility clustering** -- visible in the return plots as alternating calm and
  turbulent periods. ARMA-GARCH in notebook 02 is designed specifically to
  remove this feature from the innovations.

In [ ]:
# Return distribution plots

fig, axes = plt.subplots(1, len(TICKERS), figsize=(18, 4))

for ax, ticker in zip(axes, returns.columns):
    data = returns[ticker].dropna()
    ax.hist(data, bins=100, color="#2c5f8a", alpha=0.75, density=True, label="Empirical")

    # Overlay a Gaussian with same mean and std for comparison
    mu, sigma = data.mean(), data.std()
    x = np.linspace(data.min(), data.max(), 300)
    gaussian = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)
    ax.plot(x, gaussian, color="#d64e12", linewidth=1.5, label="Gaussian fit")

    ax.set_title(ticker, fontsize=11)
    ax.set_xlabel("Log return")
    ax.legend(fontsize=8)

fig.suptitle("Return Distributions vs Gaussian (2004–2024)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/01_return_distributions.png", bbox_inches="tight")
plt.show()
print("Figure saved to outputs/figures/01_return_distributions.png")

In [ ]:
# Save processed data

save_processed(prices,  "prices.parquet")
save_processed(returns, "returns.parquet")
print("\nAll processed data saved.")

In [ ]:
# Verification round trip

returns_check = load_processed("returns.parquet")
assert returns_check.shape == returns.shape, "Shape mismatch — file not saved correctly"
assert returns_check.isna().sum().sum() == 0, "NaNs found in saved returns"
print(f"Verification PASSED — {returns_check.shape[0]} periods, {returns_check.shape[1]} assets")
print(f"Date range: {returns_check.index[0].date()} to {returns_check.index[-1].date()}")

----

## Handoff to notebook 02

The following files are now saved in `data/processed/`:

- `prices.parquet` -- adjusted close prices, all tickers
- `returns.parquet` -- daily log returns, all tickers, NaN-free

**notebook 02** (`02_arma_garch.ipynb`) loads `returns.parquet` and fits
ARMA(1,1)-GARCH(1,1) in a 250-period rolling window across 1,000 assessment
periods, extracting standardised innovations for NIG fitting in notebook 03.


---